# Stage 0 — English Pretraining

Trains the `ProjectionMLP` and LoRA adapters on Flickr8k to establish visual-language alignment before multilingual fine-tuning.

## Setup

In [ ]:
import sys, os
sys.path.append("../src")

import torch
from transformers import AutoTokenizer
from models import ProjectionMLP, load_mt5_with_lora
from dataset import TRAINING_LANGUAGES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Precompute BLIP-2 Features

Run once; results cached to `flickr_features.pt`.

In [ ]:
# Run from repo root:
# python src/precompute_features.py --dataset flickr8k --output flickr_features.pt

FEATURE_FILE = "../flickr_features.pt"
assert os.path.exists(FEATURE_FILE), f"Run precompute_features.py first"

saved = torch.load(FEATURE_FILE, weights_only=False)
all_features       = saved["features"]           # (8K, 32, 768)
captions           = saved["captions"]           # 40K strings (5 per image)
caption_to_img_idx = saved["caption_to_img_idx"]
print(f"Features: {all_features.shape} | Captions: {len(captions)}")

## Build DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

class Flickr8kDataset(Dataset):
    def __init__(self, features, captions, img_indices, tokenizer, max_length=64):
        self.features   = features
        self.captions   = captions
        self.img_indices = img_indices
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        feature = self.features[self.img_indices[idx]]
        labels  = self.tokenizer(
            self.captions[idx],
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        ).input_ids.squeeze(0)
        return feature, labels

tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")
dataset   = Flickr8kDataset(all_features, captions, caption_to_img_idx, tokenizer)

n_val   = int(0.1 * len(dataset))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

## Initialise Models

In [ ]:
projection = ProjectionMLP().to(device)
mt5        = load_mt5_with_lora().to(device)

print(f"ProjectionMLP params: {sum(p.numel() for p in projection.parameters()):,}")

## Train

In [ ]:
import os
from tqdm import tqdm

EPOCHS    = 10
optimizer = torch.optim.AdamW(
    list(projection.parameters()) + list(mt5.parameters()),
    lr=1e-4, weight_decay=1e-2
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
os.makedirs("../outputs", exist_ok=True)
best_val = float("inf")

for epoch in range(EPOCHS):
    # Train
    projection.train(); mt5.train()
    train_loss = 0
    for features, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} train", leave=False):
        features, labels = features.to(device), labels.to(device)
        masked_labels = labels.clone()
        masked_labels[masked_labels == tokenizer.pad_token_id] = -100
        loss = mt5(inputs_embeds=projection(features), labels=masked_labels).loss
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(projection.parameters()) + list(mt5.parameters()), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # Validate
    projection.eval(); mt5.eval()
    val_loss = 0
    with torch.no_grad():
        for features, labels in tqdm(val_loader, desc="val", leave=False):
            features, labels = features.to(device), labels.to(device)
            masked_labels = labels.clone()
            masked_labels[masked_labels == tokenizer.pad_token_id] = -100
            val_loss += mt5(inputs_embeds=projection(features), labels=masked_labels).loss.item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    scheduler.step()
    print(f"Epoch {epoch+1:2d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save({"projection": projection.state_dict(), "lora": mt5.state_dict()},
                   "../outputs/english_best.pt")
        print(f"  ✓ Saved (val={val_loss:.4f})")

## Evaluate (CIDEr + BLEU on Flickr8k val set)

In [ ]:
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.bleu.bleu import Bleu

ckpt = torch.load("../outputs/english_best.pt", weights_only=True)
projection.load_state_dict(ckpt["projection"])
mt5.load_state_dict(ckpt["lora"])
projection.eval(); mt5.eval()

preds_dict, refs_dict = {}, {}

with torch.no_grad():
    for start in tqdm(range(0, len(val_ds), 16)):
        batch_idx = val_ds.indices[start:start+16]
        img_idx   = [caption_to_img_idx[i] for i in batch_idx]
        features  = torch.stack([all_features[i] for i in img_idx]).to(device)
        out = mt5.generate(inputs_embeds=projection(features), max_new_tokens=50, num_beams=4)
        for k, cap_idx in enumerate(batch_idx):
            pred = tokenizer.decode(out[k], skip_special_tokens=True)
            from datasets import load_dataset
            # refs: all 5 captions for this image
            img_i = caption_to_img_idx[cap_idx]
            # Use captions from dataset
            preds_dict[start+k] = [pred]
            refs_dict[start+k]  = [captions[img_i * 5 + j] for j in range(5) if img_i * 5 + j < len(captions)]

cider, _ = Cider().compute_score(refs_dict, preds_dict)
bleu, _  = Bleu(4).compute_score(refs_dict, preds_dict)
print(f"CIDEr:  {cider*100:.2f}")
print(f"BLEU-4: {bleu[3]*100:.2f}")